# Training a structured-output behavior

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pfekin/LARA/blob/main/examples/text2json/bonsai-1.7b/train_upload.ipynb)

This notebook trains one behavior on a frozen model and saves it as a folder of
a few megabytes. The companion notebook, `text2json_load`, downloads it and
reproduces the numbers without training anything.

A behavior is a low-rank correction applied between transformer blocks. The base
model is never modified, so the behavior can be removed at any time and carries a
strength you set at inference rather than at training time.

Point `BASE` at any causal language model. Nothing here assumes a particular one.

**Runtime.** Roughly 25 minutes on a T4 at 1.7B parameters.

## The task

Read a sentence, return one JSON object. The key names, the types and the rules
are all given in the prompt, so nothing is being guessed.

The work is in the values rather than the shape:

- `$1,234.50` has to come back as `123450`, a whole number of cents
- `2.5 kg` has to come back as `2500` grams
- `March 3rd, 2024` has to come back as `2024-03-03`
- `a dozen` has to come back as `12`
- a loose word like *cookware* has to be mapped onto one of six categories
- each sentence states seven facts and the prompt asks for four to six, so the
  rest are decoys

An answer counts only if every field is exactly right. Five fields at 85% each
gives 44% overall, which is why there is room here that a copy-the-words task
would not have.

Key names change on every example, drawn from a couple of hundred built out of
stems and suffixes. Thirty percent are held back and never trained on, so
scoring well on the held-out set means reading the keys from the prompt rather
than remembering a fixed set.

Structured output is a fair thing to measure. Models are unreliable at it, small
ones more so, and the tools that consume the output are unforgiving.

## Configuration

In [1]:
!pip install -q git+https://github.com/pfekin/LARA.git
!pip install -q transformers datasets accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import gc, json, math, os, random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from lara import LARA, Bank

NL = chr(10)

# ── the model ────────────────────────────────────────────────────────────────
# Any causal LM. Nothing below assumes a particular one.
BASE = "prism-ml/Bonsai-1.7B-unpacked"

# ── your own data, if you have it ────────────────────────────────────────────
# Leave these empty to use the generated records described further down. To
# benchmark on your own, write JSONL with one record per line:
#
#   {"sentence": "The kettle costs $12.50 and ships today.",
#    "schema": {"name": "string", "price_cents": "integer", "in_stock": "boolean"},
#    "target": {"name": "kettle", "price_cents": 1250, "in_stock": true}}
#
# `schema` gives the key names and types the prompt will ask for. `target` is
# the answer, and is compared field by field. Nothing else needs changing.
TRAIN_JSONL = ""
EVAL_JSONL  = ""

# ── where behaviors live ─────────────────────────────────────────────────────
# One repo, one folder per base model, because a behavior only loads onto the
# base it was trained against.
HF_USER       = "pfekin"
BEHAVIOR_REPO = f"{HF_USER}/lara-behaviors"
MODEL_SLUG    = BASE.split("/")[-1].lower()
BEHAVIOR      = "extract"

# ── sizes ────────────────────────────────────────────────────────────────────
BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE  = torch.bfloat16 if BF16 else torch.float16
GAMMAS = (0.0, 0.5, 1.0, 1.5)     # 0.0 is the untouched base; past 1.0 to find the peak

N_TRAIN, N_EVAL          = 1200, 120
MAX_LEN, MAX_NEW, GEN_BS = 448, 160, 8
STEPS, LR                = 1200, 2e-4
LAYERS, RANK, ALPHA      = 6, 128, 128

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{torch.cuda.get_device_name(0)}  {gb:.0f} GB  bf16={'yes' if BF16 else 'no'}")
else:
    print("no GPU: pick a GPU runtime")
print(f"base   {BASE}")
print(f"repo   {BEHAVIOR_REPO}/{MODEL_SLUG}/{BEHAVIOR}")
print(f"data   {TRAIN_JSONL or 'generated'}")

NVIDIA L4  24 GB  bf16=yes
base   prism-ml/Bonsai-1.7B-unpacked
repo   pfekin/lara-behaviors/bonsai-1.7b-unpacked/extract
data   generated


## 1. What the weights look like

Not required, but it says how the base is stored, which decides the footprint
numbers later. Nothing in the training path depends on it: the correction is
computed in floating point whatever the weights are.

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files
from safetensors import safe_open

def weight_family(repo):
    """Low-bit models are often published as their small weights written into a
    wider container. One bit per weight means a block of 128 holds two distinct
    values; ternary holds three; an ordinary model holds 128."""
    shards = sorted(f for f in list_repo_files(repo) if f.endswith(".safetensors"))
    if not shards:
        return "unknown", 16.0
    with safe_open(hf_hub_download(repo, shards[0]), framework="np") as f:
        key = next((k for k in f.keys() if k.endswith("gate_proj.weight")), None)
        if key is None:
            return "unknown", 16.0
        row = f.get_tensor(key)[0].astype(np.float32)
    counts = {len(np.unique(row[i*128:(i+1)*128])) for i in range(16)}
    zeros = float((row[:2048] == 0.0).mean())
    if counts <= {1, 2} and zeros == 0:
        return "1-bit", 1.125
    if counts <= {1, 2, 3} and zeros > 0:
        return "ternary", 2.0
    return "full precision", 16.0


FAMILY, BITS = weight_family(BASE)
print(f"{BASE}: {FAMILY}  (~{BITS} bits per weight as shipped)")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

prism-ml/Bonsai-1.7B-unpacked: 1-bit  (~1.125 bits per weight as shipped)


## 2. Data

Generated here, so every answer has exact ground truth and the run repeats. To
use your own records instead, set `TRAIN_JSONL` in the configuration cell; the
format is documented there.

Two evaluation sets. The first varies the key names. The second varies the key
names *and* the way the sentence is written, so `Nov 7 2025` instead of
`November 7, 2025` and `pantry` instead of `cookware`. Numbers on the second set
are the ones worth quoting.

In [4]:
# ── the fields a record can have ─────────────────────────────────────────────
# Each one names a type and a rule, both of which go into the prompt. The rules
# are where the work is: the sentence says "$1,234.50" and the answer has to be
# 123450, so getting a field right means reading it and converting it.
FIELDS = {
    "item":     ("string",  "the product name exactly as written"),
    "category": ("string",  "one of kitchen, lighting, outdoor, office, audio, garden"),
    "price":    ("integer", "the price in cents, with no decimal point"),
    "weight":   ("integer", "the weight in grams"),
    "date":     ("string",  "the listing date as YYYY-MM-DD"),
    "quantity": ("integer", "how many units"),
    "stock":    ("boolean", "whether it is available now"),
}

CATEGORIES = ["kitchen", "lighting", "outdoor", "office", "audio", "garden"]
SYN   = {"kitchen": ["cookware", "culinary"], "lighting": ["illumination", "lamp"],
         "outdoor": ["camping", "trail"], "office": ["desk", "workspace"],
         "audio": ["hi-fi", "sound"], "garden": ["horticulture", "yard"]}
NOVEL = {"kitchen": ["pantry", "galley"], "lighting": ["luminaire", "bulb"],
         "outdoor": ["backcountry", "alpine"], "office": ["clerical", "bureau"],
         "audio": ["acoustic", "stereo"], "garden": ["allotment", "botanical"]}
ADJ  = ["compact", "brushed", "matte", "insulated", "folding", "ceramic",
        "wireless", "reinforced", "vintage", "modular"]
NOUN = ["kettle", "lamp", "backpack", "monitor", "chair", "speaker",
        "keyboard", "bottle", "planter", "toolkit"]
MON  = ["January", "February", "March", "April", "May", "June", "July",
        "August", "September", "October", "November", "December"]
WORD = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 12: "twelve"}

# ── key names ────────────────────────────────────────────────────────────────
# Built from stems and suffixes, which gives a couple of hundred. The list is
# shuffled once and split, so training and evaluation use disjoint names. With a
# fixed set of keys a model can memorise them; with two hundred it has to read
# them out of the prompt, which is the thing being measured.
STEM = {"item": ["item", "product", "article", "goods", "listing", "entry", "piece",
                 "unit", "object", "thing"],
        "category": ["category", "dept", "section", "class", "kind", "type", "group",
                     "bucket", "family", "division"],
        "price": ["price", "cost", "amount", "value", "total", "charge", "fee", "rate"],
        "weight": ["weight", "mass", "net", "gross", "load", "shipping", "package"],
        "date": ["date", "listed", "posted", "added", "created", "registered",
                 "filed", "logged"],
        "quantity": ["quantity", "count", "units", "number", "pieces", "qty",
                     "volume", "tally"],
        "stock": ["stock", "available", "ready", "onhand", "sellable", "stocked",
                  "instock", "live"]}
SUFF = {"item": ["", "_name", "_title", "_label", "_text"],
        "category": ["", "_name", "_code", "_label"],
        "price": ["_cents", "_in_cents", "_cents_total"],
        "weight": ["_g", "_grams", "_in_grams"],
        "date": ["", "_on", "_date", "_at"],
        "quantity": ["", "_count", "_total"],
        "stock": ["", "_flag", "_now"]}

_r = random.Random(7)
POOL = {}
for _f in STEM:
    _v = sorted({a + b for a in STEM[_f] for b in SUFF[_f]})
    _r.shuffle(_v)
    POOL[_f] = _v
TRAIN_KEYS = {f: v[:int(len(v) * 0.7)] for f, v in POOL.items()}
HELD_KEYS  = {f: v[int(len(v) * 0.7):] for f, v in POOL.items()}


def instruction(schema):
    """schema maps a field to the key name the answer should use."""
    lines = [f"- {k} ({FIELDS[f][0]}): {FIELDS[f][1]}" for f, k in schema.items()]
    return ("Read the sentence and reply with one JSON object and nothing else."
            + NL + "Use exactly these keys:" + NL + NL.join(lines) + NL + NL)


def a_record(rng, novel):
    """Seven facts, written out in words. `novel` switches to phrasings the
    behavior never trains on: 'Nov 7 2025' rather than 'November 7, 2025',
    '2.4 kilos' rather than '2.4 kg', 'pantry' rather than 'cookware'."""
    syn = NOVEL if novel else SYN
    item, cat = f"{rng.choice(ADJ)} {rng.choice(NOUN)}", rng.choice(CATEGORIES)
    cents = rng.randrange(199, 400000)
    grams = rng.choice([rng.randrange(50, 990), rng.randrange(1, 40) * 100])
    y, m, d = rng.randrange(2019, 2026), rng.randrange(1, 13), rng.randrange(1, 29)
    qty = rng.choice([2, 3, 6, 12, rng.randrange(13, 60)])
    stock = rng.random() > 0.4

    if novel:
        price_t = rng.choice([f"{cents//100:,}.{cents%100:02d} USD",
                              f"{cents//100:,} dollars and {cents%100:02d} cents"])
        weight_t = (f"{grams/1000:g} kilos" if grams >= 1000 and grams % 100 == 0
                    else f"{grams} grams")
        date_t = rng.choice([f"{MON[m-1][:3]} {d} {y}", f"{d:02d}-{MON[m-1][:3]}-{y}"])
        qty_t = {2: "a couple", 6: "half a dozen", 12: "a dozen"}.get(qty, str(qty))
        stock_t = "Ships today." if stock else "On back order."
        price_c, date_c = f"Cost is {price_t}.", f"Listed {date_t}."
    else:
        price_t = rng.choice([f"${cents//100:,}.{cents%100:02d}",
                              f"{cents//100:,}.{cents%100:02d} dollars"])
        weight_t = (f"{grams/1000:g} kg" if grams >= 1000 and grams % 100 == 0
                    else f"{grams} g")
        date_t = rng.choice([f"{MON[m-1]} {d}, {y}", f"{d} {MON[m-1]} {y}"])
        qty_t = {2: "a pair", 12: "a dozen"}.get(qty, WORD.get(qty, str(qty)))
        stock_t = "It is in stock." if stock else "It is back-ordered."
        price_c, date_c = f"The price is {price_t}.", f"It was listed on {date_t}."

    clauses = [f"The {item} is a {rng.choice(syn[cat])} product.", date_c,
               f"It weighs {weight_t}.", price_c,
               f"There are {qty_t} in the shipment.", stock_t]
    rng.shuffle(clauses)
    truth = {"item": item, "category": cat, "price": cents, "weight": grams,
             "date": f"{y}-{m:02d}-{d:02d}", "quantity": qty, "stock": stock}
    return " ".join(clauses), truth


def generate_examples(rng, keys, n, novel=False):
    """An example is (prompt, target JSON, schema, truth). Four to six fields
    are asked for out of seven, so a couple of the stated facts are decoys."""
    out = []
    for _ in range(n):
        fields = rng.sample(list(POOL), rng.randint(4, 6))
        rng.shuffle(fields)
        schema = {f: rng.choice(keys[f]) for f in fields}
        sentence, truth = a_record(rng, novel)
        target = {schema[f]: truth[f] for f in schema}
        out.append((instruction(schema) + sentence, json.dumps(target), schema, truth))
    return out


def read_jsonl(path):
    """Your own data. Each line gives a sentence, the keys to ask for, and the
    answer. Types come from FIELDS by key name where they match, and default to
    whatever the target value already is."""
    out = []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        schema = {k: k for k in r["schema"]}
        lines = [f"- {k} ({t}): as stated in the sentence"
                 for k, t in r["schema"].items()]
        prompt = ("Read the sentence and reply with one JSON object and nothing else."
                  + NL + "Use exactly these keys:" + NL + NL.join(lines) + NL + NL
                  + r["sentence"])
        out.append((prompt, json.dumps(r["target"]), schema, r["target"]))
    return out


def compare(obj, schema, truth):
    """Field by field, exact. A near miss is a miss: the point of asking for
    JSON is that something downstream will read it."""
    per = {}
    for f, k in schema.items():
        got, want = obj.get(k, "__absent__"), truth[f]
        if want is None:
            ok = got is None
        elif isinstance(want, bool):
            ok = isinstance(got, bool) and got == want
        elif isinstance(want, (int, float)):
            ok = (isinstance(got, (int, float)) and not isinstance(got, bool)
                  and float(got) == float(want))
        else:
            ok = isinstance(got, str) and got.strip() == str(want)
        per[f] = bool(ok)
    return per


rng = random.Random(0)
if TRAIN_JSONL:
    train_ex = read_jsonl(TRAIN_JSONL)
    eval_seen = eval_novel = read_jsonl(EVAL_JSONL or TRAIN_JSONL)
    print(f"  {len(train_ex)} train / {len(eval_seen)} eval, from your files")
else:
    train_ex   = generate_examples(rng, TRAIN_KEYS, N_TRAIN)
    rng2 = random.Random(99)
    eval_seen  = generate_examples(rng2, HELD_KEYS, N_EVAL)
    eval_novel = generate_examples(rng2, HELD_KEYS, N_EVAL, novel=True)
    print(f"  {len(train_ex)} train")
    print(f"  {len(eval_seen)} eval, key names never trained on")
    print(f"  {len(eval_novel)} eval, key names AND phrasings never trained on")

# The grader has to accept a perfect answer, or every number below is wrong.
assert all(all(compare(json.loads(t), s, tr).values())
           for _, t, s, tr in train_ex[:200] + eval_seen + eval_novel), \
    "ground truth does not grade as correct: the comparison is broken"
print("  ground truth grades clean")

  1200 train
  120 eval, key names never trained on
  120 eval, key names AND phrasings never trained on
  ground truth grades clean


## 3. Generation and scoring

In [5]:
def load_base():
    tok = AutoTokenizer.from_pretrained(BASE)
    tok.pad_token = tok.pad_token or tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE, device_map="auto")
    return m, tok


def prompt_for(tok, text):
    msgs = [{"role": "user", "content": text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def generate(model, tok, prompts, bs=None):
    """Greedy, batched. Left padding, because generation continues from the
    right-hand end of each sequence."""
    tok.padding_side = "left"
    out = []
    for i in range(0, len(prompts), bs or GEN_BS):
        chunk = prompts[i:i + (bs or GEN_BS)]
        enc = tok(chunk, return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(model.device)
        g = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        out += [tok.decode(r[enc.input_ids.shape[1]:], skip_special_tokens=True)
                for r in g]
    return out


def evaluate(model, tok, examples):
    """Returns the share of answers correct in every field, the share that used
    the key names asked for, and the per-field accuracy behind those."""
    outs = generate(model, tok, [prompt_for(tok, p) for p, _, _, _ in examples])
    n_json = n_keys = n_all = 0
    hit, tot = {}, {}
    for text, (_, _, schema, truth) in zip(outs, examples):
        try:
            obj = json.loads(text.strip())
            assert isinstance(obj, dict)
        except Exception:
            for f in schema:
                tot[f] = tot.get(f, 0) + 1
            continue
        n_json += 1
        keys_ok = set(obj) == set(schema.values())
        n_keys += int(keys_ok)
        per = compare(obj, schema, truth)
        for f, ok in per.items():
            hit[f] = hit.get(f, 0) + int(ok)
            tot[f] = tot.get(f, 0) + 1
        n_all += int(keys_ok and all(per.values()))
    n = len(examples)
    return {"exact": n_all / n, "json": n_json / n, "keys": n_keys / n,
            "fields": {f: hit.get(f, 0) / tot[f] for f in tot},
            "outputs": outs}

In [6]:
def sweep(model, tok, bank, sets):
    """Every evaluation set at every strength, on one loaded model.

    bank=None measures the bare base. Otherwise the behavior is pinned at each
    strength in turn, so the model is loaded once rather than once per setting."""
    res = {}
    for label, examples in sets.items():
        for g in GAMMAS:
            if bank is None:
                res[(label, g)] = evaluate(model, tok, examples)
            else:
                with bank.pin({BEHAVIOR: g}):
                    res[(label, g)] = evaluate(model, tok, examples)
        line = "  ".join(f"{g}: {res[(label, g)]['exact']:.0%}" for g in GAMMAS)
        print(f"  {label:<28}{line}")
    return res


def headline(res, baseline, sets):
    print(f"{'evaluation set':<30}{'base':>8}"
          + "".join(f"{'g=' + str(g):>8}" for g in GAMMAS))
    print("-" * (38 + 8 * len(GAMMAS)))
    for label in sets:
        print(f"{label:<30}{baseline[label]['exact']:>8.0%}"
              + "".join(f"{res[(label, g)]['exact']:>8.0%}" for g in GAMMAS))
    print()
    print("Every field has to be exactly right for an answer to count.")
    drift = max(abs(res[(l, 0.0)]["exact"] - baseline[l]["exact"]) for l in sets)
    print(f"largest gap between the bare base and strength 0: {drift:.1%}")
    print("strength 0 reproduces the base: the correction is scaled to nothing")


def per_field(res, baseline, label, best):
    """Side by side, base against behavior. Which fields the base drops tells
    you more than the headline does: name-copying usually survives, and the
    fields needing a conversion or a judgement usually do not."""
    a, b = baseline[label], res[(label, best)]
    print(f"{label}, per field")
    print()
    print(f"  {'field':<12}{'base':>8}{'behavior':>10}{'change':>9}   rule")
    print("  " + "-" * 68)
    for f in FIELDS:
        if f not in a["fields"]:
            continue
        x, y = a["fields"][f], b["fields"].get(f, 0.0)
        print(f"  {f:<12}{x:>8.0%}{y:>10.0%}{y - x:>+9.0%}   {FIELDS[f][1][:34]}")
    print()
    print(f"  {'valid JSON':<12}{a['json']:>8.0%}{b['json']:>10.0%}")
    print(f"  {'right keys':<12}{a['keys']:>8.0%}{b['keys']:>10.0%}")


def show_example(res, baseline, examples, label, best, i=0):
    prompt, target, schema, truth = examples[i]
    print(prompt.split(NL + NL)[1])
    print()
    for name, r in (("base", baseline[label]), ("behavior", res[(label, best)])):
        text = r["outputs"][i].strip().replace(NL, " ")
        try:
            per = compare(json.loads(text), schema, truth)
            mark = f"{sum(per.values())}/{len(per)} fields"
        except Exception:
            mark = "not valid JSON"
        print(f"{name:<10} [{mark}]")
        print("   ", text[:230])
        print()
    print("wanted")
    print("   ", target)

## 4. Train

Three lines belong to LARA and are marked. The rest is ordinary Hugging Face
training: the modules attach to the frozen model and everything else is frozen,
so any trainer picks up the right parameters.

Loss is on the answer only. Labelling the prompt too would spend gradient on
predicting the key names in the question instead of copying them into the
answer, which is most of what this behavior has to learn.

In [7]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

model, tok = load_base()
model.config.use_cache = False

# ── LARA 1 of 3: attach. The base is frozen from here on.
lara = LARA(model, layers=LAYERS, rank=RANK, alpha=ALPHA)
print(f"{lara.num_trainable():,} trainable, base frozen")

rows = []
for prompt, target, _, _ in train_ex:
    p = tok(prompt_for(tok, prompt), add_special_tokens=False).input_ids
    f = tok(prompt_for(tok, prompt) + target + tok.eos_token,
            add_special_tokens=False).input_ids[:MAX_LEN]
    k = min(len(p), len(f))
    rows.append({"input_ids": f, "labels": [-100] * k + f[k:]})

Trainer(model=model,
        args=TrainingArguments(
            output_dir="runs/extract", max_steps=STEPS, learning_rate=LR,
            per_device_train_batch_size=1, gradient_accumulation_steps=4,
            gradient_checkpointing=True, logging_steps=300, save_strategy="no",
            bf16=BF16, fp16=not BF16, report_to=[]),
        train_dataset=Dataset.from_list(rows),
        data_collator=DataCollatorForSeq2Seq(tok, label_pad_token_id=-100)).train()

# ── LARA 2 of 3: save. Route samples are the text the behavior produces,
#    because that is what a router reads when several behaviors share a model.
lara.save(f"behaviors/{BEHAVIOR}",
          route_samples=[t for _, t, _, _ in train_ex[:200]], method="ce")
lara.detach()
model.gradient_checkpointing_disable()
model.config.use_cache = True
print(f"saved behaviors/{BEHAVIOR}")

config.json:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.40k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

3,182,592 trainable, base frozen


Step,Training Loss
300,0.008367
600,0.000223
900,0.000117
1200,0.000090


saved behaviors/extract


## 5. What it does

The model is already loaded, so the behavior is attached to it and pinned at each
strength in turn rather than reloading. Strength 0 is the check: the correction
is scaled to nothing, so it should measure identically to the bare base.

In [8]:
model.eval()
sets = {"held-out key names": eval_seen, "held-out names and wording": eval_novel}

print("bare base:")
baseline = {}
for label, ex in sets.items():
    baseline[label] = evaluate(model, tok, ex)
    print(f"  {label:<28}{baseline[label]['exact']:.0%} exact")

bank = Bank(model, tok)
bank.add(BEHAVIOR, f"behaviors/{BEHAVIOR}")     # ── LARA 3 of 3: strength at call time
print()
print("with the behavior, at each strength:")
res = sweep(model, tok, bank, sets)

print()
headline(res, baseline, sets)

bare base:
  held-out key names          39% exact
  held-out names and wording  18% exact

with the behavior, at each strength:
  held-out key names          0.0: 39%  0.5: 95%  1.0: 100%  1.5: 99%
  held-out names and wording  0.0: 18%  0.5: 57%  1.0: 66%  1.5: 50%

evaluation set                    base   g=0.0   g=0.5   g=1.0   g=1.5
----------------------------------------------------------------------
held-out key names                 39%     39%     95%    100%     99%
held-out names and wording         18%     18%     57%     66%     50%

Every field has to be exactly right for an answer to count.
largest gap between the bare base and strength 0: 0.0%
strength 0 reproduces the base: the correction is scaled to nothing


## 6. Which fields moved

The headline says how much changed. This says where. Base and behavior side by
side, so a field the base already handled is visible as such.

In [9]:
label = "held-out names and wording"
best = max(GAMMAS, key=lambda g: res[(label, g)]["exact"])
per_field(res, baseline, label, best)
print()
show_example(res, baseline, sets[label], label, best)

held-out names and wording, per field

  field           base  behavior   change   rule
  --------------------------------------------------------------------
  item             73%      100%     +27%   the product name exactly as writte
  category         23%       81%     +58%   one of kitchen, lighting, outdoor,
  price            67%       73%      +6%   the price in cents, with no decima
  weight           95%       99%      +3%   the weight in grams
  date             84%       98%     +14%   the listing date as YYYY-MM-DD
  quantity         73%       97%     +24%   how many units
  stock            77%       99%     +22%   whether it is available now

  valid JSON      100%      100%
  right keys      100%      100%

There are 39 in the shipment. Cost is 3,230.82 USD. The folding lamp is a backcountry product. Listed 04-Jul-2020. On back order. It weighs 472 grams.

base       [4/6 fields]
    {   "available_flag": true,   "goods_text": "folding lamp",   "registered_at": "04-Jul

## 7. Publish

One repo, one folder per base model, because a behavior only loads onto the base
it was trained against. `text2json_load` reads the same constants and finds it.

A write token is read from the environment, from `HF_TOKEN` in Colab's secrets
panel, or asked for once.

In [10]:
UPLOAD = True

if UPLOAD:
    from huggingface_hub import HfApi, get_token, login

    if get_token() is None:
        login()
    api = HfApi()
    print("uploading as", api.whoami()["name"])   # fails early if the token is read-only
    api.create_repo(BEHAVIOR_REPO, repo_type="model", exist_ok=True)

    mb = sum(os.path.getsize(os.path.join(d, f))
             for d, _, fs in os.walk(f"behaviors/{BEHAVIOR}") for f in fs) / 1e6
    card = ["---", "library_name: lara", "license: apache-2.0",
            "tags:", "  - lara", "  - adapter", "  - behavior",
            f"base_model: {BASE}", "---", "",
            f"# LARA behaviors for `{BASE}`", "",
            f"Base: `{BASE}` ({FAMILY}). Each folder is a behavior: a low-rank",
            "correction applied between transformer blocks. The base is never",
            "modified, and strength 0 reproduces it exactly.", "",
            "| behavior | objective | size | what it does |", "|---|---|---|---|",
            f"| `{BEHAVIOR}` | CE | {mb:.1f} MB | sentence to JSON, "
            f"following the key names and types given in the prompt |", "",
            "```python", "from lara import Bank", "bank = Bank(model, tok)",
            f'bank.add("{BEHAVIOR}", "{MODEL_SLUG}/{BEHAVIOR}")', "```", "",
            "https://github.com/pfekin/LARA · https://arxiv.org/abs/2607.28669"]
    open("behaviors/README.md", "w").write(NL.join(card))

    # Replaces this behavior's folder and leaves other models in the repo alone.
    api.upload_folder(folder_path="behaviors", repo_id=BEHAVIOR_REPO,
                      path_in_repo=MODEL_SLUG, repo_type="model",
                      commit_message=f"{BEHAVIOR} for {BASE}",
                      delete_patterns=[f"{MODEL_SLUG}/{BEHAVIOR}/*"])
    print(f"pushed to https://huggingface.co/{BEHAVIOR_REPO}/tree/main/{MODEL_SLUG}")
    print("now run text2json_load with the same BASE")

uploading as pfekin
pushed to https://huggingface.co/pfekin/lara-behaviors/tree/main/bonsai-1.7b-unpacked
now run text2json_load with the same BASE


## Notes

Strength 0 reproducing the base is a property of adding a scaled correction to
weights that were never touched, not an approximation that happens to be close.

If the held-out numbers are far below the trained ones, the behavior learned the
key names rather than the task. The share of answers using the requested keys,
printed in the per-field table, separates those two cases.

The generated sentences are tidier than real records. Anyone benchmarking this
properly should point `TRAIN_JSONL` at their own data, which is why the loader
is there.

- https://github.com/pfekin/LARA
- https://arxiv.org/abs/2607.28669